# 动态 Shape 执行优化技术

4.3 节我们看到动态 Shape 的痛点：shape 在执行期才确定，Host 每次都要重做 InferShape / Tiling / 内存分配，无法整图下沉，容易变成 Host Bound。本节的优化思路是：**把「能提前枚举的 shape」重新变回静态 Shape，让它重新享受下沉与静态优化的全部收益。** 这就是 **动态分档（Dynamic Gear）**。

本节学习大纲如下：

- 优化总思路：分档 > 完全动态
- 动态分档的三种模式与配置（atc / Session）
- 执行期：怎么选档、设档（aclmdl 接口）
- 稳定兜底策略：分档图 + 动态图渐进降级
- 动态 shape 范围与其他运行时优化
- 收益验证：profiling 对比分档前后
- 小结

## 1. 优化总思路：分档 > 完全动态

动态 Shape 的优化优先级，从高到低：

<p align="center"><img src="./images/dynamic_optimization_priority.svg" alt="动态 Shape 优化优先级" width="85%"></p>

**动态分档的本质**：在编译期一次性枚举所有可能出现的输入 shape 组合（称为「档位 / gear」），为每个档位生成独立的**静态优化子图**；运行时根据实际输入 shape 选择对应子图执行。这样每个档位都能享受静态 Shape 的全部编译优化（算子融合、内存规划、下沉调度），同时保留有限的动态灵活性。

代价也要心里有数：

| 代价 | 说明 |
| --- | --- |
| 内存按最大档位计 | 即使执行最小档，内存占用仍等同最大档位 |
| 编译时间随档位线性增长 | N 个档位 ≈ N 倍单一静态 Shape 编译时间 |
| 未命中档位会失败 | 实际 shape 不匹配任何档位时，需由业务侧转到动态图兜底 |

> **CANN 9.0 + 910B 范围**：三种动态分档的档位数取值均为 `(1, 100]`，即至少 2 档、最多 100 档；其中 `--dynamic_dims` 官方建议配置 3～4 档。动态 batch / 动态分辨率也应根据真实流量选择少量高频档位。不同产品和后续版本的上限可能不同，不能把 100 当成跨版本固定值。

## 2. 动态分档的三种模式与配置

GE 提供三种分档模式，按变化维度选择：

| 模式 | atc 参数 | 适用 | 限制 |
| --- | --- | --- | --- |
| 动态 Batch | `--dynamic_batch_size` | 仅 batch（N）维变化 | `-1` 只能在第一维；与其他两种互斥 |
| 动态分辨率 | `--dynamic_image_size` | 仅 H/W 维变化 | H 和 W 必须同时变；与其他两种互斥 |
| 任意维度动态 | `--dynamic_dims` | 任意多个维度变化 | 最灵活，配置最复杂；可覆盖前两种 |

### 2.1 离线编译（atc）

```shell
# 动态 batch：input_shape 用 -1 标记动态维，dynamic_batch_size 枚举档位
atc --model=resnet50.onnx --framework=5 --output=resnet50_dyn_batch \
    --soc_version=Ascend910B1 \
    --input_shape="input:-1,3,224,224" \
    --dynamic_batch_size="1,8,16"      # 只允许 batch=1/8/16 三档

# 动态分辨率：HW 同时变
atc --model=det.onnx --framework=5 --output=det_dyn_hw \
    --soc_version=Ascend910B1 --input_format=NCHW \
    --input_shape="input:8,3,-1,-1" \
    --dynamic_image_size="416,416;832,832"   # 支持 416x416 / 832x832

# 任意维度动态：每组对应一个档位中所有 -1 维的具体取值
atc --model=m.onnx --framework=5 --output=m_dyn_dims \
    --soc_version=Ascend910B1 --input_format=ND \
    --input_shape="data:1,1,40,-1;label:1,-1;mask:-1,-1" \
    --dynamic_dims="20,20,1,1;40,40,2,2;80,60,4,4"
```

> `--dynamic_dims` 中每个分号分隔的一组值，依次对应 `--input_shape` 里所有 `-1` 的取值。上例三档分别为：data(1,1,40,20)/label(1,20)/mask(1,1)，data(1,1,40,40)/label(1,40)/mask(2,2)，data(1,1,40,80)/label(1,60)/mask(4,4)。

### 2.2 在线编译（GeSession / aclgrphBuildModel）

在线路径通过 options 传入等价配置。注意离线用 `--dynamic_*`，在线用 `ge.*` / `ir_option` 选项名：

```cpp
// 在线动态维度：AddGraph / Session options
std::map<ge::AscendString, ge::AscendString> options = {
    {"ge.inputShape",      "data:1,1,40,-1;label:1,-1;mask:-1,-1"},
    {"ge.dynamicDims",     "20,20,1,1;40,40,2,2;80,60,4,4"},
    {"ge.dynamicNodeType", "1"}   // 0:dataset 输入动态；1:placeholder 输入动态
};
session.AddGraph(graph_id, graph, options);
```

```cpp
// 在线动态 batch：aclgrphBuildModel 的 ir_option
options.insert({
    {ge::ir_option::INPUT_FORMAT,       "NCHW"},          // 本例与 Data format 一致
    {ge::ir_option::INPUT_SHAPE,        "data:-1,1,28,28"},// -1 表示动态 batch
    {ge::ir_option::DYNAMIC_BATCH_SIZE, "2,4,8"}           // batch 档位
});
```

> 几个易错点：
> - **动态分辨率**依赖 layout 定位 H/W，因此 `INPUT_FORMAT` 只支持 NCHW / NHWC，并且要与 Data format 一致。
> - **动态 batch**要求动态 N 位于 shape 第一维，但该模式本身不把输入格式限制为 NCHW / NHWC；若显式设置 `INPUT_FORMAT`，仍须与 Data format 一致。
> - ATC 的 `--dynamic_dims` 用于 ND 格式下的任意维度分档，按官方用法显式配套 `--input_format=ND`。
> - 整图设置动态维度时，`ge.inputShape` 的输入顺序要与 Data 节点 name 的**字母序**一致。
> - 构图时对应的 Data 节点 shape 要把动态维设为 `-1`。

## 3. 执行期：怎么选档、设档

编译出分档模型后，**执行前要告诉模型这次用哪一档**。不同分档模式对应不同的 acl 接口：

| 分档模式 | 执行前调用的接口 | 不调用的默认行为 |
| --- | --- | --- |
| 动态 batch | `aclmdlSetDynamicBatchSize` | 按 batch 档位最大值赋值 |
| 动态分辨率 | `aclmdlSetDynamicHWSize` | 按最大档位宽高赋值 |
| 任意维度动态 | `aclmdlSetInputDynamicDims` | 按动态维度最大值赋值 |
| 动态 shape 范围 | `aclmdlSetDatasetTensorDesc` | 需显式设置真实输入 Tensor 描述 |

```c
// 离线 ACL 推理：执行前设置真实档位，再 aclmdlExecute
// 例：动态 batch，本次用 batch=8
size_t index;
aclmdlGetInputIndexByName(modelDesc, ACL_DYNAMIC_TENSOR_NAME, &index); // 获取动态档位输入下标
aclmdlSetDynamicBatchSize(modelId, input, index, 8);                   // 设置真实 batch 档位
aclmdlExecute(modelId, input, output);                                 // 执行
```

<p align="left"><img src="./images/gear_selection.svg" alt="运行时选档流程" width="65%"></p>

> 要点：**设档值必须落在编译时枚举的档位里**，否则执行失败。设档发生在「模型执行接口之前」。

### 3.1 动手实践：真实分档图与动态 fallback

本例使用应用侧显式路由：分别编译一张 batch=1/8/16 的分档图和一张完全动态图，不依赖自动 Hybrid 路由。请求 batch=5 时补齐到 8 并走分档图；batch=20 超出所有档位时，代码显式转到动态图兜底。

四种请求都会真实送入 0 号 NPU 并与 NumPy 对拍。路由逻辑在代码中清晰可见，不会把未命中的 shape 误送给分档 Session。

> **语义边界**：本例只有逐元素 ReLU，各 batch 样本互不影响，因此补零到下一档后再裁剪输出不会改变真实样本结果。包含 batch 维归约、跨样本交互、状态更新或依赖有效长度的模型，不能直接套用这种 padding；除非模型正确使用 mask 且结果经过验证，否则应直接走动态图兜底。

> **耗时提示**：首次执行分档图和动态图时都会触发编译，整个单元可能需要数分钟；请根据 `[INFO]` 提示等待当前步骤完成。


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning)
# === 真机运行：动态档位 + padding 命中 + 独立动态图兜底 ===
import time

import numpy as np
from ge.es.graph_builder import GraphBuilder
from ge.es.nn import Relu
from ge.ge_global import GeApi
from ge.graph import Tensor
from ge.graph.types import DataType, Format
from ge.session import Session

DEVICE_ID = 0
GEAR_GRAPH_ID = 1
DYNAMIC_GRAPH_ID = 2
FEATURES = 3
GEARS = (1, 8, 16)
# 本图只有逐元素 ReLU，已确认 padding 不会影响真实样本；其他模型必须重新评估。
PADDING_IS_SEMANTICALLY_SAFE = True


def _output_to_numpy(tensor, expected_shape):
    try:
        return np.asarray(tensor.data, dtype=np.float32).reshape(expected_shape)
    except (ValueError, TypeError):
        import ctypes
        import ge.graph.tensor as _tm
        c_str = _tm.graph_lib.GeApiWrapper_Tensor_GetData(tensor._handle)
        try:
            data_str = ctypes.string_at(c_str).decode("utf-8")
            flat = _tm._parse_str_list(data_str)
        finally:
            _tm.graph_lib.GeApiWrapper_FreeString(c_str)
        return np.asarray(flat, dtype=np.float32).reshape(-1, expected_shape[-1])

gear_options = {
    "ge.inputShape": "input_x:-1,3",
    "ge.dynamicDims": "1;8;16",
    "ge.dynamicNodeType": "1",
}


def build_relu_graph(name):
    builder = GraphBuilder(name)
    x = builder.create_input(
        index=0,
        name="input_x",
        data_type=DataType.DT_FLOAT,
        shape=[-1, FEATURES],
    )
    builder.set_graph_output(Relu(x), 0)
    return builder.build_and_reset()


def prepare_request(input_array, allow_padding):
    batch = input_array.shape[0]
    selected = next((gear for gear in GEARS if gear >= batch), None)
    if selected is None:
        return input_array, "dynamic_fallback", None
    if selected == batch:
        return input_array, "exact_gear", selected
    if not allow_padding:
        return input_array, "dynamic_fallback", None

    pad_rows = selected - batch
    padded = np.pad(input_array, ((0, pad_rows), (0, 0)))
    return padded, "padded_gear", selected


ge_api = GeApi()
ge_api.ge_initialize({
    "ge.exec.deviceId": str(DEVICE_ID),
    "ge.graphRunMode": "0",
    "ge.exec.precision_mode_v2": "origin",
})
gear_session = None
dynamic_session = None
outputs = None
input_tensor = None
try:
    # 应用侧显式路由：分档 Session 处理高频 shape，动态 Session 负责未命中档位的输入。
    gear_session = Session(gear_options)
    dynamic_session = Session()
    print("[INFO] 正在准备 batch=1/8/16 分档图与动态 fallback 图", flush=True)
    gear_session.add_graph(
        GEAR_GRAPH_ID, build_relu_graph("DynamicGearGraph"), gear_options
    )
    dynamic_session.add_graph(
        DYNAMIC_GRAPH_ID, build_relu_graph("DynamicFallbackGraph")
    )

    for requested_batch in (1, 5, 16, 20):
        request = np.linspace(
            -2.0, 2.0, num=requested_batch * FEATURES, dtype=np.float32
        ).reshape(requested_batch, FEATURES)
        model_input, route, selected_gear = prepare_request(
            request, PADDING_IS_SEMANTICALLY_SAFE
        )
        model_batch = model_input.shape[0]

        input_tensor = Tensor(
            model_input.reshape(-1).tolist(),
            None,
            DataType.DT_FLOAT,
            Format.FORMAT_ND,
            [model_batch, FEATURES],
        )
        # 没有可用档位或不能安全 padding 时，不进入分档图，显式切换到动态图。
        if route == "dynamic_fallback":
            active_session = dynamic_session
            active_graph_id = DYNAMIC_GRAPH_ID
            print(
                f"[INFO] batch={requested_batch} 无可用的安全分档路径，正在执行动态图 fallback",
                flush=True,
            )
        else:
            active_session = gear_session
            active_graph_id = GEAR_GRAPH_ID

        started = time.perf_counter()
        outputs = active_session.run_graph(active_graph_id, [input_tensor])
        elapsed_ms = (time.perf_counter() - started) * 1e3

        # padding 路径只保留真实请求对应的输出。
        actual = _output_to_numpy(outputs[0], (model_batch, FEATURES))[:requested_batch]
        expected = np.maximum(request, 0.0)
        np.testing.assert_allclose(actual, expected, rtol=1e-6, atol=1e-6)

        print(
            "request_batch={:<2d} model_batch={:<2d} route={:<16s} gear={} NPU E2E={:.3f} ms".format(
                requested_batch,
                model_batch,
                route,
                selected_gear,
                elapsed_ms,
            )
        )

    print("[OK] 精确档位、padding 档位与独立动态图兜底均已在 NPU 上验证")
finally:
    outputs = None
    input_tensor = None
    # 释放 Session 引用，由 Session 析构统一释放图资源。
    dynamic_session = None
    gear_session = None
    ge_api.ge_finalize()


## 4. 稳定兜底：分档 Session + 动态 Session

纯分档有个硬伤：**未命中任何档位就执行失败**。可采用双 Session 方案：应用同时准备分档图和动态图，根据输入 shape 显式选择执行路径。

- **分档图（gear graph）**：带 `inputShape + dynamicDims`，编成 Case + N 个静态子图。
- **动态 shape 图（dynamic shape graph）**：去掉分档约束，编成真正的动态 Shape 图。

运行时先尝试精确命中档位；仅当 padding 对模型语义安全时，才补齐到下一档，否则直接选择动态图：

<p align="left"><img src="./images/hybrid_mode.svg" alt="分档图与动态图兜底路径" width="50%"></p>

核心代码如下：

```python
gear_session = Session(gear_options)  # 高频 shape：分档优化
dynamic_session = Session()           # 非预期 shape：动态图兜底

if selected_gear is None:
    outputs = dynamic_session.run_graph(DYNAMIC_GRAPH_ID, inputs)
else:
    outputs = gear_session.run_graph(GEAR_GRAPH_ID, inputs)
```

> Padding 不是通用兜底：它要求不同 batch 样本之间没有相互作用，且补齐数据不会参与影响真实样本的归约、状态或序列计算；padding-sensitive 模型还必须正确传递 mask / 有效长度并裁剪输出。本节 ReLU 示例满足这些条件。
>
> 这种方式优先使用分档图的静态性能，不能安全 padding 或超出最大档位时用动态图保证正确性。代价是需要编译并持有两张图。

## 5. 动态 shape 范围与其他运行时优化

### 5.1 动态输入 shape 范围

当 shape 连续可变、难以离散成有限档位时，可在编译期指定**输入 shape 范围**（用 `~` 表示区间，`-1` 表示无限定）：

```cpp
// 在线：指定 N 维 8~20、最后一维 -1（无限定）
options.insert({
    {ge::ir_option::INPUT_FORMAT, "NCHW"},
    {ge::ir_option::INPUT_SHAPE,  "8~20,3,5,-1"}
});
// 执行前用 aclmdlSetDatasetTensorDesc 设置真实输入 Tensor 描述，
// 执行后用 aclmdlGetDatasetTensorDesc 获取动态输出描述
```

格式要点：按 name `"in1:8~20,3,5,-1;in2:5,3~9,10,-1"` 或按 index 设置（需给 Data 设 index 属性，从 0 开始）。

> shape 范围比分档更灵活，但仍是动态 Shape 路径（Host 调度）。范围约束可帮助编译器限定合法 shape 和资源上界，实际性能取决于算子与编译结果，不能预设一定介于分档和完全无约束动态之间，应以 profiling 为准。

### 5.2 其他降 Host 开销的手段

| 手段 | 思路 |
| --- | --- |
| AICPU 与 AI Core 并行 | 动态 Shape 流分配较保守；将编译选项 `AC_PARALLEL_ENABLE` 设为 `1` 后，编译器会识别可与 AI Core 并发的 AICPU 算子，默认值为 `0` |
| 收敛输入 shape 多样性 | 业务侧把请求归一到少数高频 shape；仅在模型语义允许时使用 padding，让分档更容易命中 |
| 复用执行器 / Session | 同一 Session 多次执行复用已加载资源，避免重复加载与初始化 |
| 外部 allocator / 内存池 | 复用内存，减少每次执行的内存申请/释放开销（注意流同步后再释放）|

`AC_PARALLEL_ENABLE` 是**图编译选项**，不是环境变量。在线构图和 ATC 分别这样设置：

```cpp
options.insert({{ge::ir_option::AC_PARALLEL_ENABLE, "1"}});
```

```shell
atc ... --ac_parallel_enable=1
```

> 开启该选项不保证所有 AICPU / AI Core 算子都能并行，是否分流取决于图依赖和编译器识别结果，仍需通过 timeline 验证。

> 动态 Shape 流分配比静态保守：默认单流，Data/Variable/NetOutput 等强制在主流，仅 Event 同步。能并行的空间有限，所以**分档仍是收益最大的手段**。

## 6. 收益验证：profiling 对比分档前后

分档到底把 Host 开销压下去多少，用 profiling 量化。

### 6.1 采集

```shell
# 应用内部先完成相同次数的 warm-up，再以相同输入 shape、相同迭代数采集两种路径
msprof --application="./infer_dynamic" --output=/tmp/prof_dyn  --ge-api=l1 --runtime-api=on --task-time=on --aicpu=on --ai-core=on
msprof --application="./infer_geared"  --output=/tmp/prof_gear --ge-api=l1 --runtime-api=on --task-time=on --aicpu=on --ai-core=on
```

### 6.2 对比指标

| 指标 | 完全动态 | 分档命中（期望）|
| --- | --- | --- |
| Host 侧 InferShape/Tiling/内存分配耗时 | 高 | **大幅下降**（编译期已完成）|
| 单次执行下发 Task 数 | 多（逐算子）| **少**（命中档位走静态子图、可下沉）|
| Device timeline 空泡 | 多 | **少**（更连续）|
| 单次 E2E 耗时 | 基线 | **下降**，接近同 shape 静态 |
| AI Core 利用率 | 低 | 上升 |

```
 分档前（完全动态，Host Bound）：
   Host  : ▮Infer▮Tiling▮Alloc▮发▮ ......
   Device:                  ▮算▮ ▯空泡▯ ▮算▮
 分档后（命中档位→静态子图→下沉）：
   Host  : ▮（少量/单模型 Task）▮
   Device: ▮▮▮▮▮▮▮▮（连续）
```

> `--ge-api=l1` 用于采集动态 shape 的细粒度 Host 调度数据（包括 InferShape / Tiling 等阶段）；`--runtime-api=on` 用于定位 Runtime 调用、同步和 memcpy；其余选项用于关联 Task、AICPU 与 AI Core 数据。
>
> 验证闭环：先确认动态版本是 Host Bound（4.3 节方法），再确认分档请求确实命中目标档位，并比较相同预热、输入与迭代口径。若分档命中后 Device 仍有空泡，应结合 GE API 与 Runtime timeline 排查 Host 调度、同步依赖、数据搬运、AICPU 和过小 Task；只有 Device 持续繁忙且算子执行占主导时，才可判断为 Device Bound。

## 7. 小结

- 动态 Shape 优化的核心是 **动态分档**：编译期枚举档位、每档生成静态优化子图，命中档位即重获下沉与静态优化收益。
- 三种分档模式：`--dynamic_batch_size`（仅 batch）、`--dynamic_image_size`（仅 HW）、`--dynamic_dims`（任意维，最灵活），三者互斥；在线对应 `ge.inputShape + ge.dynamicDims + ge.dynamicNodeType` 等选项。
- 执行前需**设档**：`aclmdlSetDynamicBatchSize` / `aclmdlSetDynamicHWSize` / `aclmdlSetInputDynamicDims`；不设则按最大档位赋值，设错（未命中）则失败。
- CANN 9.0 可用分档 Session + 动态 Session 实现稳定兜底：命中走分档图；只有语义安全时才 padding 到下一档，否则未命中输入直接走动态图。
- shape 连续可变时可用**动态 shape 范围**（`~`）；其余手段如 AICPU/AICore 并行、收敛 shape、复用资源可进一步降 Host 开销。
- 收益用 **profiling 对比分档前后** 的 Host 耗时、下发 Task 数、Device 空泡、E2E 与 AI Core 利用情况；采集 Host 动态调度需开启 `--ge-api`。

> 本章四节走完：静态执行（下沉）→ 动态执行（Host 调度）→ 静态优化（内存/并发）→ 动态优化（分档）。下一节用综合练习把它们串起来。

## 课后练习

完成下列题目自测，如有错误建议结合本节对应小节复盘。

1. （判断题）动态分档在编译期为每个档位生成独立的静态优化子图，命中档位时可享受静态 Shape 的编译优化与下沉。

2. （判断题）使用动态分档后，即使执行最小档位，模型内存占用仍等同于最大档位。

3. （判断题）动态 batch、动态分辨率、任意维度动态三种分档模式可以同时使用。

4. （单选题）以下哪个 atc 参数用于配置「仅 batch 维度变化」的动态分档？
    A. `--dynamic_image_size`
    B. `--dynamic_dims`
    C. `--dynamic_batch_size`
    D. `--input_format`

5. （单选题）离线推理时，动态 batch 模型在执行前设置真实 batch 档位应调用哪个接口？
    A. `aclmdlSetDynamicHWSize`
    B. `aclmdlSetDynamicBatchSize`
    C. `aclmdlSetInputDynamicDims`
    D. `aclmdlSetDatasetTensorDesc`

6. （单选题）分档 Session + 动态 Session 相比只有分档图的主要优势是？
    A. 编译更快
    B. 内存占用更小
    C. 输入未命中任何档位时可降级到动态 shape 图执行，不会失败
    D. 不需要枚举档位

7. （多选题）以下关于动态 Shape 优化的描述，哪些是正确的？
    A. 动态分档是把可枚举的 shape 重新变回静态 Shape 以重获优化收益
    B. 设置的档位值若未命中任何编译档位，且没有动态图兜底，执行会失败
    C. 动态 shape 范围用 `~` 表示区间、`-1` 表示无限定，仍属动态 Shape 路径
    D. 动态分档可以无限增加档位且不增加编译时间和内存

**执行以下代码获取答案。**

In [ ]:
!cat ./answer/04.05_answer.txt